# Notebook 02: Exploratory Data Analysis (EDA)
This notebook performs Exploratory Data Analysis on the cleaned Walmart Store Sales dataset. The goal is to uncover patterns, trends, seasonality, and correlations that will guide feature engineering and modeling.

## Objectives:
1. **Analyze Sales Trends**: Visualize weekly sales over time to understand overall patterns and seasonality.
2. **Store Type & Size Influence**: Investigate how store type and size relate to sales performance.
3. **Holiday Impact**: Examine if holiday weeks show significant sales spikes compared to regular weeks.
4. **Feature Correlation**: Check correlations between sales and numerical features (temperature, fuel price, CPI, unemployment, and markdowns).

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# Add the project root to python path
sys.path.append(os.path.abspath('..'))

## 1. Load Cleaned Dataset

In [ ]:
from src import config

# Read the cleaned training data
df = pd.read_csv(config.TRAIN_CLEANED_PATH)
df['Date'] = pd.to_datetime(df['Date'])
df.head()

## 2. Overall Weekly Sales Trend
Let's aggregate sales by date to see the trend over the entire time series (2010 to 2012).

In [ ]:
timeline_sales = df.groupby('Date')['Weekly_Sales'].sum().reset_index()

plt.figure(figsize=(15, 6))
plt.plot(timeline_sales['Date'], timeline_sales['Weekly_Sales'], color='royalblue', linewidth=2, marker='o', markersize=4)
plt.title('Total Weekly Sales Across All Stores (2010 - 2012)', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Total Sales ($)', fontsize=12)
plt.axvline(pd.to_datetime('2010-11-26'), color='red', linestyle='--', alpha=0.7, label='Thanksgiving 2010')
plt.axvline(pd.to_datetime('2010-12-24'), color='darkred', linestyle='--', alpha=0.7, label='Christmas 2010')
plt.axvline(pd.to_datetime('2011-11-25'), color='red', linestyle='--', alpha=0.7, label='Thanksgiving 2011')
plt.axvline(pd.to_datetime('2011-12-23'), color='darkred', linestyle='--', alpha=0.7, label='Christmas 2011')
plt.legend(frameon=True)
plt.tight_layout()
plt.show()

### Observation:
* There are very clear annual spikes around **Thanksgiving (late November)** and **Christmas (late December)**.
* Sales drop significantly in January immediately following the holiday season.

## 3. Store Type & Size Analysis
Let's look at how store Type (A, B, or C) and Size influence weekly sales.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Boxplot of Sales by Store Type
sns.boxplot(ax=axes[0], data=df, x='Type', y='Weekly_Sales', showfliers=False, palette='muted')
axes[0].set_title('Weekly Sales Distribution by Store Type', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Store Type')
axes[0].set_ylabel('Weekly Sales ($)')

# Plot 2: Store Size Distribution by Type
sns.boxplot(ax=axes[1], data=df.drop_duplicates(subset=['Store']), x='Type', y='Size', palette='muted')
axes[1].set_title('Store Size Distribution by Store Type', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Store Type')
axes[1].set_ylabel('Store Size (sq ft)')

plt.tight_layout()
plt.show()

### Observation:
* **Type A** stores are the largest and have the highest median weekly sales.
* **Type C** stores are the smallest and have the lowest weekly sales.

## 4. Holiday vs. Non-Holiday Sales
Let's check if the difference in sales during holiday weeks vs. non-holiday weeks is statistically noticeable.

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=df, x='IsHoliday', y='Weekly_Sales', ci=95, palette='Set2')
plt.title('Average Weekly Sales: Holiday vs. Non-Holiday Weeks', fontsize=14, fontweight='bold')
plt.xlabel('Is Holiday Week')
plt.ylabel('Average Weekly Sales ($)')
plt.show()

### Observation:
* Weeks containing major holidays have higher average weekly sales than non-holiday weeks.

## 5. Month-of-Year Seasonality
Let's look at average weekly sales grouped by Month to find broader monthly trends.

In [ ]:
df['Month'] = df['Date'].dt.month
monthly_sales = df.groupby('Month')['Weekly_Sales'].mean().reset_index()

plt.figure(figsize=(12, 6))
sns.barplot(data=monthly_sales, x='Month', y='Weekly_Sales', palette='Blues_d')
plt.title('Average Weekly Sales by Month', fontsize=14, fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Average Weekly Sales ($)')
plt.show()

### Observation:
* **December** is the peak sales month due to holiday shopping.
* **November** is the second highest (Thanksgiving and Black Friday).
* **January** experiences the lowest sales of the year.

## 6. Correlation Analysis
Let's check the correlation between the numerical features and the target variable (`Weekly_Sales`).

In [ ]:
cols_to_corr = ['Weekly_Sales', 'Size', 'Temperature', 'Fuel_Price', 
                'CPI', 'Unemployment', 'IsHoliday', 
                'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']

corr_matrix = df[cols_to_corr].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.5)
plt.title('Correlation Matrix of Sales and Features', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### Observation:
* **Size** has the strongest positive correlation with `Weekly_Sales` (~0.24).
* The MarkDown indicators have small positive correlations.
* Features like `Temperature`, `Fuel_Price`, `CPI`, and `Unemployment` show very weak linear correlations with sales, suggesting their relationships are non-linear or depend heavily on store/dept context (meaning we will need non-linear models like GBDTs or neural networks).